In [3]:
import os
from dotenv import load_dotenv
from pathlib import Path
from ai_engineer_preproject import chunk_text, VectorStore, VecIdxSearch
load_dotenv()
# Получаем путь к папке с документами (данные для LLM)
PROJECT_ROOT = Path(os.getenv("PROJECT_ROOT"))
DOCS_DIR = Path(os.getenv("OLLAMA_DOCS"))
OLLAMA_URL = os.getenv("OLLAMA_URL")

In [5]:
try:
    search_engine = VecIdxSearch(
        data_dir=str(DOCS_DIR),
        index_factory="Flat",
        # Можно передать кастомный набор расширений, если нужно:
        extensions={'.md', '.mdx', '.txt'} 
    )
    print("Индекс успешно построен. Поиск готов.")
except ValueError as e:
    print(f"Ошибка при инициализации: {e}")
    # Если здесь ошибка, поиск не сработает
    search_engine = None
print(f"Загружено документов: {search_engine.num_documents}")
print(f"Получено чанков: {search_engine.num_chunks}")   

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Индекс успешно построен. Поиск готов.
Загружено документов: 70
Получено чанков: 609


In [6]:
if search_engine:
    question = "Как сменить папку, в которой Ollama хранит модели?"
    
    # top_k=3 означает, что мы хотим получить 5 наиболее релевантных фрагментов
    results = search_engine.search(question=question, top_k=3)
for i, r in enumerate(results, 1):
    print(f"#{i}  [{r['source']}]  score={r['score']:.3f}")
    print(r["text"][:200], "...\n")

#1  [openai-compatibility.mdx]  score=1.400
```python vision.py
from openai import OpenAI

client = OpenAI(
    base_url='http://localhost:11434/v1/',
    api_key='ollama',  # required but ignored
) ...

#2  [openai-compatibility.mdx]  score=1.437
```javascript vision.js
import OpenAI from "openai";

const openai = new OpenAI({
  baseURL: "http://localhost:11434/v1/",
  apiKey: "ollama", // required but ignored
}); ...

#3  [chatgpt.mdx]  score=1.453
## Prerequisites

- Ollama v0.34.0 or newer

If ChatGPT is not installed, Ollama offers to download it during setup. ...



In [7]:
# --- Использование ---

# 1. Подготовка данных (один раз при старте приложения)
EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
# chunks = [...]
# index = faiss.IndexIDMap(...)

# 2. Создание экземпляра (инъекция зависимостей)
vector_db = VectorStore(
    embed_model_name=EMBED_MODEL,
    index=index,
    chunks=chunks
)

# 3. Выполнение поиска
results = vector_db.search("Как сменить папку, в которой Ollama хранит модели?", top_k=5)

NameError: name 'index' is not defined

In [ ]:
print(f"Запрос: {question}\n")
for i, r in enumerate(results, 1):
    print(f"#{i}  [{r['source']}]  score={r['score']:.3f}")
    print(r["text"][:200], "...\n")



In [ ]:
questions = [
    "What HTTP method does /api/generate use?",
    "Как сменить папку, где хранятся модели?",
    "What is a Modelfile?",
]

for q in questions:
    print(f"=== {q} ===")
    results = vector_db.search(q, top_k=5) 
    for r in results:
        print(f"  [{r['source']}]  score={r['score']:.3f}")
        print(f"  {r['text'][:150].strip()}...\n") 

